# Packaging and Transferring Files to Archivematica

This workflow assumes you have a package already in this structure, using one of the starting points at our main [GitHub site](https://github.com/mlibrary/digiPres/tree/main/workflows):

```
📂 barcode
├── 📂 carved_files
│   └── carved files
└── 📂 transfer_metadata
    └── checksum file
```

## Establishing File Paths
Start by entering the path to top-level barcode directory.

In [ ]:
path_to_barcode_directory = input("Enter path to barcode directory: ").strip("'")
path_to_carved_files = path_to_barcode_directory + "/carved_files"
path_to_transfer_metadata = path_to_barcode_directory + "/transfer_metadata"

print("top level directory:", path_to_barcode_directory)
print("carved_files:", path_to_carved_files)
print("transfer_metadata:", path_to_transfer_metadata)

## 🌳 Generate File Tree

In [ ]:
from directory_tree import DisplayTree

tree_file = path_to_transfer_metadata + "/tree.txt"

with open(tree_file, "w", encoding="utf-8") as t:
    t.write(DisplayTree(path_to_carved_files, showHidden=True, stringRep=True))
    t.close

print("tree file at", tree_file)

## 📋 Generate Brunnhilde Reports

In [ ]:
import subprocess
import sys

module_name = "brunnhilde"
no_virus = "-n"
source = path_to_carved_files
destination = path_to_transfer_metadata + "/brunnhilde_reports"

try:
    result = subprocess.run(
        [sys.executable, "-m", "brunnhilde", no_virus, source, destination],
        capture_output=True,
        text=True,
        check=True
    )
    print("Module Output:", result.stdout)

except subprocess.CalledProcessError as e:
    print(f"The module failed with exit code {e.returncode}")
    print("Error output:\n", e.stderr)

## 📝 Metadata

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

# Create the form input widgets
barcode_input = widgets.Text(placeholder='', description='BarcodeNumberIdentifier')
title_input = widgets.Text(placeholder='', description='Title')
alternateTitle_input = widgets.Text(placeholder='', description='AlternateTitle')
accessionNumberCollection_input = widgets.Text(placeholder='', description='AccessionNumberCollection')
originatingUnit_input = widgets.Text(placeholder='', description='OriginatingUnitDepartment')
retentionPriority_dropdown = widgets.Dropdown(
    options=['', 'Low', 'Medium', 'High'], 
    value='', 
    description='MediaRetentionPriority'
)
recordsLabel_input = widgets.Text(placeholder='', description='RecordsLabel')
mediumCarrier_dropdown = widgets.Dropdown(
    options=['', 'CD', 'DVD', 'HDD/Hard-Drive', 'USB', 'Floppy (3.5in)', 'Floppy (5.25in)', 'Floppy (8in)', 'Zip Disk', 'Jazz Drive', 'Tape/Disk Cartridge', 'Memory Card (SD/HC/XC/UHS/MS PRO/PRO Duo/Compact Flash/etc.)', 'Magneto-Optical Disc (1990s)', 'Mobile Device (Phone/Tablet/etc.)', 'Computer', 'Folder on Computer', 'Email', 'Content Systems (RMS/DMS/RDMS/CMS Repository)', 'Network Storage (NAS/SAN/FTP/SFTP)', 'Cloud Storage', 'Other (Describe in Notes)'], 
    value='', 
    description='MediumCarrier'
)
conditionNotes_input = widgets.Text(placeholder='', description='MediumCarrierConditionNotes')
conditionDetails_input = widgets.Text(placeholder='', description='MediumCarrierDetails')
mediumCarrier2_dropdown = widgets.Dropdown(
    options=['', 'CD', 'DVD', 'HDD/Hard-Drive', 'USB', 'Floppy (3.5in)', 'Floppy (5.25in)', 'Floppy (8in)', 'Zip Disk', 'Jazz Drive', 'Tape/Disk Cartridge', 'Memory Card (SD/HC/XC/UHS/MS PRO/PRO Duo/Compact Flash/etc.)', 'Magneto-Optical Disc (1990s)', 'Mobile Device (Phone/Tablet/etc.)', 'Computer', 'Folder on Computer', 'Email', 'Content Systems (RMS/DMS/RDMS/CMS Repository)', 'Network Storage (NAS/SAN/FTP/SFTP)', 'Cloud Storage', 'Other (Describe in Notes)'], 
    value='', 
    description='MediumCarrier2'
)
conditionNotes2_input = widgets.Text(placeholder='', description='MediumCarrierConditionNotes2')
conditionDetails2_input = widgets.Text(placeholder='', description='MediumCarrierDetails2')
hardware_dropdown = widgets.Dropdown(
    options=['', 'External 3.5in Floppy Reader', 'Internal 3.5in Floppy Reader on vintage machine', 'Internal Optical Drive', 'USB via WriteBlocker', 'Direct USB', 'File Hosting Service', 'FC5025', 'Applesauce', 'KryoFlux'], 
    value='', 
    description='Hardware'
)
software_dropdown = widgets.Dropdown(
    options=['', 'FTK Imager', 'Data Accessioner', 'Disc Copy on vintage machine', 'FC5025 Disk Image and Browse', 'KryoFlux Host Software', 'Applesauce', 'Other (Describe in Notes)'], 
    value='', 
    description='Software'
)
writeBlockerHardware_dropdown = widgets.Dropdown(
    options=['', 'Tableau USB Bridge', 'Tableau FireWire Bridge', 'Floppy Disk Write Protect', 'Read-Only Optical Disc', 'Applesauce Safe Switch'], 
    value='', 
    description='WriteBlockerHardware'
)
virusCheckRun_checkbox = widgets.Checkbox(value=False, description='VirusCheckRun')
virusCheckPassed_checkbox = widgets.Checkbox(value=False, description='VirusCheckPassed')
virusCheckResultsNotes_input = widgets.Text(placeholder='', description='VirusCheckResultsNotes')
bagContent_dropdown = widgets.Dropdown(
    options=['', 'Disk Image', 'Logical Transfer', 'Stream Files'], 
    value='', 
    description='BagContent'
)
notes_input = widgets.Text(placeholder='', description='Notes')
bagCreator_input = widgets.Text(placeholder='', description='BagCreator')
submit_button = widgets.Button(description='Submit & Save', button_style='success', icon='save')

# Create an output widget to display confirmation status
output_area = widgets.Output()

# Define the submission and file saving logic
def on_submit_clicked(b):
    filename = path_to_transfer_metadata + "/metadata.txt"

    submission_text = {}

    for widget in form_layout.children:
        print(widget.description)
        submission_text[widget.description] = str(widget.value)
     
    try:
        with open(filename, "w", encoding="utf-8") as file:
            file.write(json.dumps(submission_text, indent=2))
            
        with output_area:
            output_area.clear_output()
            print(f"💾 Metadata successfully saved to '{filename}'!")
            
    except Exception as e:
        with output_area:
            output_area.clear_output()
            print(f"❌ Error saving file: {e}")

# Link the button to the function
submit_button.on_click(on_submit_clicked)

# Display the layout
form_layout = widgets.VBox([
    barcode_input, 
    title_input, 
    alternateTitle_input, 
    accessionNumberCollection_input,
    originatingUnit_input,
    retentionPriority_dropdown,
    recordsLabel_input,
    mediumCarrier_dropdown,
    conditionNotes_input,
    conditionDetails_input,
    mediumCarrier2_dropdown,
    conditionNotes2_input,
    conditionDetails2_input,
    hardware_dropdown,
    software_dropdown,
    writeBlockerHardware_dropdown,
    virusCheckRun_checkbox,
    virusCheckPassed_checkbox,
    virusCheckResultsNotes_input,
    bagContent_dropdown,
    notes_input,
    bagCreator_input
])

display(form_layout, submit_button, output_area)


Note: check that everything is there!
```
📂 barcode
├── 📂 carved_files
│   └── carved files
└── 📂 transfer_metadata
    └── checksums.txt or csv
    └── tree.txt
    └── 📂 brunnhilde_reports
    └── metadata.txt
```

## Bagit and Checksum Validation

Run this cell and follow the prompts to batch bag, validate checksums, and/or create a globus ingest file. This cell pulls directly from the Bagit and Checksum Validation (BAC) Python script. Prompt options include:
- ```'b'``` to bag packages and automatically run checksum validation. You will be prompted afterwards on whether or not to generate a globus file for this batch.
- ```'c'``` to run checksum validation on a bag or batch of bags.
- ```'g'``` to generate a globus search ingest file for a bag or batch of bags.

In [ ]:
import BAC

path_to_metadata_file = path_to_transfer_metadata + "/metadata.txt"

BAC.single_bag(path_to_barcode_directory, path_to_metadata_file)